## 📝 Exercises

### Task 1
Parse text data into structured format

---

### Task 2
Count occurrences of each name

---

### Task 3
Filter invalid records safely

---

### Task 4
Calculate the average salary per department

---

### Task 5
Calculate the number of employees for each department

In [7]:
import os
import sys

# 1. إجبار البيئة على استخدام بايثون الصحيح لمنع مشاكل التواصل
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"

# 2. إضافة البادئة file:// لإجبار سبارك على القراءة المحلية من المجلد
file_path = "file:///data/spark-assignments/01-RDD-Lab-Employees/employees.txt"
raw_rdd = sc.textFile(file_path)

# 3. محاولة استخراج الترويسة وتنظيف البيانات
try:
    header = raw_rdd.first()
    clean_lines_rdd = raw_rdd.filter(lambda line: line != header and line.strip() != "")
    
    print("✓ Success! File read completed from Local File System.")
    print(f"Total raw lines in file: {raw_rdd.count()}")
    print(f"Total data rows to process: {clean_lines_rdd.count()}")
except Exception as e:
    print("An error occurred:")
    print(e)

✓ Success! File read completed from Local File System.
Total raw lines in file: 13
Total data rows to process: 10


In [4]:
print("Current Working Directory inside Container:", os.getcwd())
print("Files in this directory:", os.listdir('.'))

Current Working Directory inside Container: /data/spark-assignments
Files in this directory: ['01-RDD-Lab-Employees', 'docker-compose.yml']


### Task 1: Parse text data into structured format

We transform the semi-structured comma-separated text lines into a structured format by parsing each line into a list of individual fields using the `split(',')` function.

In [9]:
# Map each line to a split list of strings based on the comma delimiter
parsed_rdd = clean_lines_rdd.map(lambda line: line.split(','))

# Display the first 3 structured records to verify transformation
print("Sample of parsed structured data:")
for record in parsed_rdd.take(3):
    print(record)

Sample of parsed structured data:
['1', 'John Smith', 'Engineering', 'Senior Developer', '125000', 'San Francisco', '2021-03-15', '4.5', '8']
['2', 'Sarah Johnson', 'Sales', 'Account Executive', '85000', 'New York', '2022-01-10', '4.2', '5']
['3', 'Michael Williams', 'Engineering', 'Software Engineer', '95000', 'Austin', '2023-06-20', '3.8', '3']


### Task 2: Count occurrences of each name

To verify the frequency of each employee's name across the dataset, we transform each structured record into a key-value pair formatted as `(name, 1)`. We then apply the `reduceByKey` transformation to aggregate and compute the total occurrences for each unique name.

In [10]:
# Map each record to a tuple of (name, 1) and aggregate the counts by name
name_counts_rdd = parsed_rdd.map(lambda fields: (fields[1].strip(), 1)) \
                           .reduceByKey(lambda a, b: a + b)

print("Employee Name Occurrences:")
for name, count in name_counts_rdd.collect():
    print(f" - {name}: {count}")

Employee Name Occurrences:
 - Sarah Johnson: 1
 - Michael Williams: 1
 - Jennifer Brown: 1
 - David Jones: 1
 - Robert Martinez: 1
 - Patricia Wilson: 1
 - James Anderson: 1
 - Mary Thomas: 1
 - John Smith: 1
 - Lisa Garcia: 1


### Task 3: Filter invalid records safely

Real-world datasets often contain malformed data. In this step, we implement a data quality check using a custom validation function. This ensures each record contains exactly 9 fields and that the salary column (index 4) can be parsed into a numeric float safely. Malformed records are isolated into an audit RDD for tracing and analysis, preventing downstream computation failures.

In [11]:
def validate_employee_record(fields):
    """
    Validates the structure of an employee record:
    1. Ensures the row has exactly 9 elements.
    2. Verifies that the salary field (index 4) can be parsed into a float.
    """
    if len(fields) != 9:
        return False
    try:
        float(fields[4].strip())
        return True
    except (ValueError, IndexError):
        return False

# Filter for valid data to be used in analytics
valid_rdd = parsed_rdd.filter(validate_employee_record)

# Isolate corrupted rows for data auditing and debugging
invalid_rdd = parsed_rdd.filter(lambda fields: not validate_employee_record(fields))

print(f"Successfully processed valid records: {valid_rdd.count()}")
print(f"Corrupted records isolated: {invalid_rdd.count()}")

print("\n--- Isolated Corrupted Records Audit ---")
for bad_record in invalid_rdd.collect():
    print(f"Malformed Row: {bad_record}")

Successfully processed valid records: 9
Corrupted records isolated: 1

--- Isolated Corrupted Records Audit ---
Malformed Row: ['7', 'Robert Martinez', 'Legal', 'Legal Counsel145000', 'San Francisco', '2019-09-22', '4.8', '10', ' 5']


### Task 4: Calculate the average salary per department

To extract financial insights per business unit, we map the validated records into a key-value structure formatted as `(department, (salary, 1))`. We aggregate both the total salaries and the employee counts simultaneously using the `reduceByKey` transformation, and finally, we calculate the mathematical average for each department.

In [12]:
# Map to (department, (salary, 1)) using clean and verified records
dept_salary_mapper = valid_rdd.map(lambda fields: (fields[2].strip(), (float(fields[4].strip()), 1)))

# Reduce to aggregate total salary sum and total count per department
dept_salary_totals = dept_salary_mapper.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Map values to calculate the average: total_salary / total_count
dept_avg_salary = dept_salary_totals.mapValues(lambda totals: totals[0] / totals[1])

print("Average Salary Per Department:")
for dept, avg_sal in dept_avg_salary.collect():
    print(f" - {dept}: ${avg_sal:,.2f}")

Average Salary Per Department:
 - Sales: $97,500.00
 - Finance: $105,000.00
 - Engineering: $121,666.67
 - Marketing: $92,000.00
 - IT: $115,000.00
 - HR: $88,000.00


### Task 5: Calculate the number of employees for each department

To analyze the workforce distribution across the organization, we map each validated employee record into a key-value pair formatted as `(department, 1)`. We then apply the `reduceByKey` transformation to aggregate the counts, providing an accurate headcount for each individual department.

In [13]:
# Map to (department, 1) and aggregate the headcount using reduceByKey
dept_headcount_rdd = valid_rdd.map(lambda fields: (fields[2].strip(), 1)) \
                              .reduceByKey(lambda a, b: a + b)

print("Employee Headcount Per Department:")
for dept, count in dept_headcount_rdd.collect():
    print(f" - {dept}: {count} employee(s)")

Employee Headcount Per Department:
 - Sales: 2 employee(s)
 - Finance: 1 employee(s)
 - Engineering: 3 employee(s)
 - Marketing: 1 employee(s)
 - IT: 1 employee(s)
 - HR: 1 employee(s)
